In [1]:
import mysql.connector
import sqlite3

In [2]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Souravmysql1946",
    database="inventory"
)

In [3]:
def connected_to_db():
    return mysql.connector.connect(
    host="localhost",
    user="root",
    password="Souravmysql1946",
    database="inventory"
)

In [4]:
connected_to_db().is_connected()

True

In [4]:
db = connected_to_db()
cursor = db.cursor(dictionary=True)

In [5]:
import bcrypt

def hash_password(password: str) -> bytes:
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt())

hashed = hash_password("admin123")
cursor.execute(
    "INSERT INTO users (username, password_hash, role) VALUES (%s, %s, %s)",
    ("admin", hashed.decode(), "Admin")
)
db.commit()

In [7]:
cursor.execute("select count(*) as total_suppliers from suppliers")

In [8]:
cursor.fetchall()

[{'total_suppliers': 50}]

In [12]:
queries = {
"Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

"Total Products": "SELECT COUNT(*) AS count FROM products",

"Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

"Total Sale Value (Last 3 Months)": """
SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) AS total_sale
FROM stock_entries se
JOIN products p ON se.product_id = p.product_id
WHERE se.change_type = 'Sale'
AND se.entry_date >= (
SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries where change_type='Sale')
""",

"Total Restock Value (Last 3 Months)": """
SELECT ROUND(SUM(se.change_quantity * p.price), 2) AS total_restock
FROM stock_entries se
JOIN products p ON se.product_id = p.product_id
WHERE se.change_type = 'Restock'
AND se.entry_date >= (
SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries where change_type='Restock')
""",

"Below Reorder & No Pending Reorders": """
SELECT COUNT(*) AS below_reorder
FROM products p
WHERE p.stock_quantity < p.reorder_level
AND p.product_id NOT IN (
SELECT DISTINCT product_id FROM reorders WHERE status = 'Pending')
"""
}

In [14]:
result={}
for label, query in queries.items():
    cursor.execute(query)
    row= cursor.fetchone()
    result[label]=list(row.values())[0]
    

In [15]:
result

{'Total Suppliers': 50,
 'Total Products': 205,
 'Total Categories Dealing': 6,
 'Total Sale Value (Last 3 Months)': 1529482.99,
 'Total Restock Value (Last 3 Months)': 120799.5,
 'Below Reorder & No Pending Reorders': 14}

In [31]:
## Now lets create function out of it 

def get_basic_info(cursor):
    """
    Retrieve summary inventory and supply chain metrics.

    Args:
        cursor (mysql.connector.cursor.MySQLCursorDict): Cursor object with dictionary=True.

    Returns:
        dict: Dictionary of metric labels and their values.
    """

    queries = {
        "Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

        "Total Products": "SELECT COUNT(*) AS count FROM products",

        "Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

        "Total Sale Value (Last 3 Months)": """
            SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) AS total_sale
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Sale'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Total Restock Value (Last 3 Months)": """
            SELECT ROUND(SUM(se.change_quantity * p.price), 2) AS total_restock
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Restock'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Below Reorder & No Pending Reorders": """
            SELECT COUNT(*) AS below_reorder
            FROM products p
            WHERE p.stock_quantity < p.reorder_level
              AND p.product_id NOT IN (
                  SELECT DISTINCT product_id FROM reorders WHERE status = 'Pending')
        """
    }

    results = {}
    for label, query in queries.items():
        cursor.execute(query)
        row = cursor.fetchone()
        # Since row is a dictionary, extract the single value by getting the first value in dict.values()
        results[label] = list(row.values())[0]

    return results



In [32]:
queries = {
        "Suppliers Contact Details": "SELECT supplier_name, contact_name, email, phone FROM suppliers",

        "Products with Supplier and Stock": """
            SELECT 
                p.product_name,
                s.supplier_name,
                p.stock_quantity,
                p.reorder_level
            FROM products p
            JOIN suppliers s ON p.supplier_id = s.supplier_id
            ORDER BY p.product_name ASC
        """,

        "Products Needing Reorder": """
            SELECT product_name, stock_quantity, reorder_level
            FROM products
            WHERE stock_quantity <= reorder_level
        """
    }

tables = {}
for label, query in queries.items():
    cursor.execute(query)
    tables[label] = cursor.fetchall()

In [33]:
def get_additonal_tables(cursor):
    queries = {
        "Suppliers Contact Details": "SELECT supplier_name, contact_name, email, phone FROM suppliers",

        "Products with Supplier and Stock": """
            SELECT 
                p.product_name,
                s.supplier_name,
                p.stock_quantity,
                p.reorder_level
            FROM products p
            JOIN suppliers s ON p.supplier_id = s.supplier_id
            ORDER BY p.product_name ASC
        """,

        "Products Needing Reorder": """
            SELECT product_name, stock_quantity, reorder_level
            FROM products
            WHERE stock_quantity <= reorder_level
        """
    }

    tables = {}
    for label, query in queries.items():
        cursor.execute(query)
        tables[label] = cursor.fetchall()

    return tables

In [37]:
def add_new_manual_id(cursor, db, p_name, p_category, p_price, p_stock, p_reorder, p_supplier):
    proc_call = "call AddNewProductManualID(%s, %s, %s, %s, %s, %s)"
    params = (p_name, p_category, p_price, p_stock, p_reorder, p_supplier)
    cursor.execute(proc_call, params)
    db.commit()
    
add_new_manual_id(cursor, db, "Product X", "Clothing", 19.99, 50, 10, 1)

In [9]:
def get_categories(cursor):
    cursor.execute("SELECT DISTINCT category FROM products ORDER BY category ASC")
    return [row['category'] for row in cursor.fetchall()]

In [10]:
get_categories(cursor)

['Clothing', 'Electronic', 'Electronics', 'Furniture', 'Groceries', 'Toys']

In [6]:
def get_suppliers(cursor):
    cursor.execute("SELECT supplier_id, supplier_name FROM suppliers ORDER BY supplier_name ASC")
    return cursor.fetchall()

In [7]:
supplier = get_suppliers(cursor)

In [8]:
supplier[0]['supplier_name']

'Anderson-Thompson'

In [9]:
supplier

[{'supplier_id': 1, 'supplier_name': 'Anderson-Thompson'},
 {'supplier_id': 48, 'supplier_name': 'Armstrong-Vance'},
 {'supplier_id': 24, 'supplier_name': 'Barker, White and Carson'},
 {'supplier_id': 25, 'supplier_name': 'Barrett Ltd'},
 {'supplier_id': 3, 'supplier_name': 'Baxter-Meadows'},
 {'supplier_id': 19, 'supplier_name': 'Charles Inc'},
 {'supplier_id': 41, 'supplier_name': 'Clark Group'},
 {'supplier_id': 23, 'supplier_name': 'Douglas Ltd'},
 {'supplier_id': 32, 'supplier_name': 'Elliott-Ayers'},
 {'supplier_id': 7, 'supplier_name': 'Evans Inc'},
 {'supplier_id': 27, 'supplier_name': 'Franklin, Kane and Price'},
 {'supplier_id': 15, 'supplier_name': 'Freeman-Gordon'},
 {'supplier_id': 13, 'supplier_name': 'Gallagher-Miller'},
 {'supplier_id': 17, 'supplier_name': 'Gomez PLC'},
 {'supplier_id': 40, 'supplier_name': 'Hall-Brown'},
 {'supplier_id': 38, 'supplier_name': 'Harris-Cummings'},
 {'supplier_id': 42, 'supplier_name': 'Henderson LLC'},
 {'supplier_id': 50, 'supplier_name

In [10]:
supplier_id = [p_supplier["supplier_id"] for p_supplier in supplier]
supplier_name = [p_supplier["supplier_name"] for p_supplier in supplier]

In [16]:
supplier_id.index(4)

47

In [ ]:
supplier_id.index()

[{'supplier_id': 1, 'supplier_name': 'Anderson-Thompson'},
 {'supplier_id': 48, 'supplier_name': 'Armstrong-Vance'},
 {'supplier_id': 24, 'supplier_name': 'Barker, White and Carson'},
 {'supplier_id': 25, 'supplier_name': 'Barrett Ltd'},
 {'supplier_id': 3, 'supplier_name': 'Baxter-Meadows'},
 {'supplier_id': 19, 'supplier_name': 'Charles Inc'},
 {'supplier_id': 41, 'supplier_name': 'Clark Group'},
 {'supplier_id': 23, 'supplier_name': 'Douglas Ltd'},
 {'supplier_id': 32, 'supplier_name': 'Elliott-Ayers'},
 {'supplier_id': 7, 'supplier_name': 'Evans Inc'},
 {'supplier_id': 27, 'supplier_name': 'Franklin, Kane and Price'},
 {'supplier_id': 15, 'supplier_name': 'Freeman-Gordon'},
 {'supplier_id': 13, 'supplier_name': 'Gallagher-Miller'},
 {'supplier_id': 17, 'supplier_name': 'Gomez PLC'},
 {'supplier_id': 40, 'supplier_name': 'Hall-Brown'},
 {'supplier_id': 38, 'supplier_name': 'Harris-Cummings'},
 {'supplier_id': 42, 'supplier_name': 'Henderson LLC'},
 {'supplier_id': 50, 'supplier_name

In [ ]:
supplier_name[supplier_id.index(4)] #index need value return index

'Wilson, Graham and Williams'

In [ ]:
supplier_id[4] 

3